In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
y = train["PitNextLap"].astype(int).values

oof_006b = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_006b_cat_original_prior_no_year_prior_min.npy"
)
oof_007 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_007_realmlp_shared001_min.npy"
)
oof_008 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_008_realmlp_shared001_plus_006b_orig_prior.npy"
)
oof_009 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_009_lgb_shared003_full_fe_single_min.npy"
)

def report(name, oof):
    print(name)
    print("  AUC:", roc_auc_score(y, oof))
    print()

report("006b_cat_orig_prior_no_year", oof_006b)
report("007_realmlp_shared001_min", oof_007)
report("008_realmlp_plus_orig_prior", oof_008)
report("009_lgb_shared003_full_fe_single", oof_009)

pairs = [
    ("006b", oof_006b, "009", oof_009),
    ("007", oof_007, "009", oof_009),
    ("008", oof_008, "009", oof_009),
]

print("\nCorrelation:")
for a_name, a, b_name, b in pairs:
    pearson = np.corrcoef(a, b)[0, 1]
    spearman = spearmanr(a, b).correlation
    print(f"{a_name} vs {b_name}: pearson={pearson:.6f}, spearman={spearman:.6f}")

blend_candidates = {
    "avg_007_009": (oof_007 + oof_009) / 2,
    "avg_007_008_009": (oof_007 + oof_008 + oof_009) / 3,
    "avg_006b_007_009": (oof_006b + oof_007 + oof_009) / 3,
    "avg_006b_007_008_009": (oof_006b + oof_007 + oof_008 + oof_009) / 4,
}

print("\nBlend probe:")
for name, o in blend_candidates.items():
    print(name, roc_auc_score(y, o))

006b_cat_orig_prior_no_year
  AUC: 0.9487943937294769

007_realmlp_shared001_min
  AUC: 0.9532701289015783

008_realmlp_plus_orig_prior
  AUC: 0.9530389769769412

009_lgb_shared003_full_fe_single
  AUC: 0.9510132993481392


Correlation:
006b vs 009: pearson=0.955461, spearman=0.963078
007 vs 009: pearson=0.956694, spearman=0.968295
008 vs 009: pearson=0.957863, spearman=0.968978

Blend probe:
avg_007_009 0.9535764080996907
avg_007_008_009 0.9538695925161512
avg_006b_007_009 0.9532941742775516
avg_006b_007_008_009 0.9536776164355024
